# Further Feature Engineering

This notebook adds advanced features to the existing 2D GP features:
1. **Physics features** (percentiles, detection, SNR)
2. **Bazin Function Fitting** (parametric light curve model)
3. **PCA on Interpolated Curves** (shape extraction)
4. **Statistical Markers** via `light-curve` library (Stetson K, Amplitude, Beyond1Std)

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.stats import skew, kurtosis
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# Try importing light_curve, install if not available
try:
    import light_curve as lc
    LIGHT_CURVE_AVAILABLE = True
except ImportError:
    print("light-curve not installed. Install with: pip install light-curve")
    LIGHT_CURVE_AVAILABLE = False

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
import sys

# Add project root to path for local imports
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config_loader import load_config, get_path, get_seed

config = load_config()
RANDOM_STATE = get_seed()

TRAIN_MODE = True  # Set to False for test data processing

# Paths from config
RAW_DATA_PATH = get_path('data_raw')
PROCESSED_DATA_PATH = get_path('data_processed')

# Input/Output files
if TRAIN_MODE:
    # Assuming upstream 2DGP features are saved with specific names
    # Adjust if your previous step output names differ
    INPUT_PARQUET = PROCESSED_DATA_PATH / 'mallorn_train_features.parquet'
    OUTPUT_PARQUET = PROCESSED_DATA_PATH / 'further_train_features.parquet'
    LOG_FILE = RAW_DATA_PATH / 'train_log.csv'
    LC_PREFIX = 'train'
else:
    INPUT_PARQUET = PROCESSED_DATA_PATH / 'mallorn_test_features.parquet'
    OUTPUT_PARQUET = PROCESSED_DATA_PATH / 'further_test_features.parquet'
    LOG_FILE = RAW_DATA_PATH / 'test_log.csv'
    LC_PREFIX = 'test'

# Physics constants
BANDS = ['u', 'g', 'r', 'i', 'z', 'y']
Z_LIMIT_TDE = 1.9

# PCA settings
PCA_TIME_GRID_START = -50  # Days relative to peak
PCA_TIME_GRID_END = 100
PCA_TIME_GRID_POINTS = 75
PCA_N_COMPONENTS = 3

print(f"Mode: {'TRAIN' if TRAIN_MODE else 'TEST'}")
print(f"Input: {INPUT_PARQUET}")
print(f"Output: {OUTPUT_PARQUET}")

In [ ]:
# Load existing processed features
print("Loading existing GP features...")
try:
    df_existing = pd.read_parquet(INPUT_PARQUET)
    print(f"Existing features shape: {df_existing.shape}")
except FileNotFoundError:
    print("Input file not found. Ensure previous step ran correctly.")
    df_existing = pd.DataFrame()

# Load metadata
print("Loading metadata...")
try:
    log_df = pd.read_csv(LOG_FILE)
    print(f"Metadata shape: {log_df.shape}")
except FileNotFoundError:
    print("Metadata log file not found.")
    log_df = pd.DataFrame()

## 1. Bazin Function Parametric Fitting

$$F(t) = A \frac{e^{-(t-t_0)/\tau_{fall}}}{1 + e^{-(t-t_0)/\tau_{rise}}} + B$$

In [ ]:
def bazin(t, A, t0, tau_fall, tau_rise, B):
    """Bazin function for light curve fitting."""
    return A * np.exp(-(t - t0) / tau_fall) / (1 + np.exp(-(t - t0) / tau_rise)) + B


def fit_bazin(times, fluxes, flux_errs):
    """
    Fit Bazin function to light curve data.
    Returns: dict with A, t0, tau_fall, tau_rise, B, and fit_failed flag
    """
    result = {
        'bazin_A': np.nan,
        'bazin_t0': np.nan,
        'bazin_tau_fall': np.nan,
        'bazin_tau_rise': np.nan,
        'bazin_B': np.nan,
        'bazin_fit_failed': 1
    }
    
    if len(times) < 5:
        return result
    
    try:
        # Sort by time
        idx = np.argsort(times)
        t = times[idx]
        f = fluxes[idx]
        
        # Initial guesses
        t_range = t.max() - t.min()
        A_init = f.max() - f.min()
        t0_init = t[np.argmax(f)]
        B_init = np.median(f)
        tau_init = t_range / 4
        
        p0 = [A_init, t0_init, tau_init, tau_init, B_init]
        
        # Bounds
        bounds = (
            [0, t.min() - 100, 0.1, 0.1, -np.inf],  # Lower bounds
            [np.inf, t.max() + 100, t_range * 2, t_range * 2, np.inf]  # Upper bounds
        )
        
        # Fit
        popt, _ = curve_fit(
            bazin, t, f, p0=p0, bounds=bounds,
            maxfev=5000, sigma=flux_errs[idx] if flux_errs is not None else None
        )
        
        result = {
            'bazin_A': popt[0],
            'bazin_t0': popt[1],
            'bazin_tau_fall': popt[2],
            'bazin_tau_rise': popt[3],
            'bazin_B': popt[4],
            'bazin_fit_failed': 0
        }
    except Exception:
        pass
    
    return result

In [ ]:
def calculate_percentile_features(fluxes, band):
    """Capture distribution shape without curve fitting."""
    if len(fluxes) == 0:
        return {}
    
    p = np.percentile(fluxes, [5, 25, 50, 75, 95])
    
    return {
        f'flux_p5_{band}': p[0],
        f'flux_p25_{band}': p[1],
        f'flux_p50_{band}': p[2],
        f'flux_p75_{band}': p[3],
        f'flux_p95_{band}': p[4],
        f'flux_iqr_{band}': p[3] - p[1],
        f'flux_range_robust_{band}': p[4] - p[0],
        f'flux_ratio_peak_median_{band}': p[4] / (p[2] + 1e-9)
    }


def calculate_detection_features(times, fluxes, errors, band, threshold=5.0):
    """Calculate statistics on significant detections (SNR > threshold)."""
    snr = fluxes / (errors + 1e-9)
    mask = snr > threshold
    
    if not np.any(mask):
        return {
            f'detected_duration_{band}': 0,
            f'detected_count_{band}': 0,
            f'detected_max_snr_{band}': 0
        }
    
    t_det = times[mask]
    return {
        f'detected_duration_{band}': t_det.max() - t_det.min(),
        f'detected_count_{band}': int(len(t_det)),
        f'detected_max_snr_{band}': float(np.max(snr[mask]))
    }


def calculate_snr_features(fluxes, errors, band):
    """Calculate SNR statistics per band."""
    if len(fluxes) == 0:
        return {}
    snr = fluxes / (errors + 1e-9)
    return {
        f'snr_max_{band}': np.max(snr),
        f'snr_mean_{band}': np.mean(snr),
        f'snr_std_{band}': np.std(snr)
    }

In [ ]:
def calculate_statistical_markers(times, fluxes, errors):
    """
    Calculate statistical markers using light-curve library.
    Falls back to manual calculation if library not available.
    """
    result = {
        'stetson_k': np.nan,
        'amplitude_robust': np.nan,
        'beyond_1_std': np.nan
    }
    
    if len(times) < 3:
        return result
    
    if LIGHT_CURVE_AVAILABLE:
        try:
            # Create extractor with desired features
            extractor = lc.Extractor(
                lc.StetsonK(),
                lc.Amplitude(),
                lc.BeyondNStd(1.0)
            )
            
            # Sort by time
            idx = np.argsort(times)
            t = np.asarray(times[idx], dtype=np.float64)
            m = np.asarray(fluxes[idx], dtype=np.float64)
            e = np.asarray(errors[idx], dtype=np.float64)
            
            # Extract features
            features = extractor(t, m, e)
            
            result['stetson_k'] = features[0]
            result['amplitude_robust'] = features[1]
            result['beyond_1_std'] = features[2]
        except Exception:
            pass
    else:
        # Manual fallback calculations
        try:
            # Stetson K (approximate)
            mean_flux = np.mean(fluxes)
            residuals = (fluxes - mean_flux) / (errors + 1e-9)
            n = len(residuals)
            result['stetson_k'] = np.sum(np.abs(residuals)) / np.sqrt(n * np.sum(residuals**2))
            
            # Amplitude (robust: 95th - 5th percentile)
            result['amplitude_robust'] = np.percentile(fluxes, 95) - np.percentile(fluxes, 5)
            
            # Beyond 1 std
            std_flux = np.std(fluxes)
            result['beyond_1_std'] = np.mean(np.abs(fluxes - mean_flux) > std_flux)
        except Exception:
            pass
    
    return result

In [ ]:
def process_object(obj_data, obj_id, z_value):
    """
    Process a single object and extract all new features.
    """
    features = {'object_id': obj_id}
    
    # Get all times and fluxes for Bazin fitting (all bands combined)
    # Note: Bazin is usually per-band, but here we try a global fit for overall shape
    # or we can iterate bands. Let's do all-bands combined relative to peak for shape.
    all_times = obj_data['Time (MJD)'].values
    all_fluxes = obj_data['Flux'].values
    all_errors = obj_data['Flux_err'].values
    
    # 1. Bazin Fitting (on all bands combined)
    bazin_feats = fit_bazin(all_times, all_fluxes, all_errors)
    features.update(bazin_feats)
    
    # 2. Statistical Markers (on all bands combined)
    stat_feats = calculate_statistical_markers(all_times, all_fluxes, all_errors)
    features.update(stat_feats)
    
    # 3. Per-band features
    for band in ['g', 'r', 'i']:  # Focus on main bands
        band_mask = obj_data['Filter'].str.contains(band, case=False, na=False)
        band_data = obj_data[band_mask]
        
        if len(band_data) < 3:
            continue
        
        t = band_data['Time (MJD)'].values
        f = band_data['Flux'].values
        e = band_data['Flux_err'].values
        
        # Percentile features
        pct_feats = calculate_percentile_features(f, band)
        features.update(pct_feats)
        
        # Detection features
        det_feats = calculate_detection_features(t, f, e, band)
        features.update(det_feats)
        
        # SNR features
        snr_feats = calculate_snr_features(f, e, band)
        features.update(snr_feats)
    
    # 4. Global features
    features['low_Z'] = 1 if (pd.notna(z_value) and z_value < Z_LIMIT_TDE) else 0
    
    # Peak color g-r
    if f'flux_p95_g' in features and f'flux_p95_r' in features:
        features['peak_color_g_r'] = features['flux_p95_g'] / (features['flux_p95_r'] + 1e-9)
    
    return features

In [ ]:
# Main processing loop - process each split separately to save memory if raw data is large
all_features = []
all_interpolated = []  # For PCA

# Determine how to load raw lightcurves. 
# If raw data is in splits, we iterate.
# Assuming standard structure: raw/split_XX/train(or test)_full_lightcurves.csv

print(f"Processing lightcurves from {RAW_DATA_PATH}...")

for split_num in tqdm(range(1, 21), desc="Processing Splits"):
    folder = f"split_{split_num:02d}"
    file_path = RAW_DATA_PATH / folder / f"{LC_PREFIX}_full_lightcurves.csv"
    
    if not file_path.exists():
        # Fallback for local dev or slightly different structure
        # Try searching if not exact match
        possible = list(RAW_DATA_PATH.glob(f"*{split_num}*/*{LC_PREFIX}*lightcurves.csv"))
        if possible:
            file_path = possible[0]
        else:
            print(f"Skipping {folder} - file not found")
            continue
    
    # Load split data
    chunk_df = pd.read_csv(file_path)
    
    # Get unique objects in this split
    object_ids = chunk_df['object_id'].unique()
    
    for obj_id in object_ids:
        obj_data = chunk_df[chunk_df['object_id'] == obj_id]
        
        # Get Z value from metadata
        z_row = log_df[log_df['object_id'] == obj_id]
        z_value = z_row['Z'].values[0] if len(z_row) > 0 else np.nan
        
        # Process object
        obj_features = process_object(obj_data, obj_id, z_value)
        all_features.append(obj_features)
        
        # Store interpolated curve for PCA (if enough data)
        times = obj_data['Time (MJD)'].values
        fluxes = obj_data['Flux'].values
        
        if len(times) >= 5:
            # Normalize time to peak
            peak_time = times[np.argmax(fluxes)]
            t_rel = times - peak_time
            
            # Create interpolation grid
            grid = np.linspace(PCA_TIME_GRID_START, PCA_TIME_GRID_END, PCA_TIME_GRID_POINTS)
            
            try:
                # Interpolate to grid
                idx = np.argsort(t_rel)
                interp_func = interp1d(
                    t_rel[idx], fluxes[idx],
                    kind='linear', bounds_error=False, fill_value=np.nan
                )
                interp_flux = interp_func(grid)
                
                # Store with object_id
                all_interpolated.append({
                    'object_id': obj_id,
                    'interp_flux': interp_flux
                })
            except Exception:
                pass
    
    # Memory cleanup after each split
    del chunk_df
    gc.collect()

print(f"\nProcessed {len(all_features)} objects")
print(f"Objects with valid interpolation: {len(all_interpolated)}")

In [ ]:
# Build PCA features
print("Extracting PCA features from interpolated curves...")

# Build matrix
pca_object_ids = [x['object_id'] for x in all_interpolated]
flux_matrix = np.array([x['interp_flux'] for x in all_interpolated])

if len(flux_matrix) > 0:
    print(f"Flux matrix shape: {flux_matrix.shape}")

    # Handle NaN values - fill with row mean
    row_means = np.nanmean(flux_matrix, axis=1, keepdims=True)
    nan_mask = np.isnan(flux_matrix)
    flux_matrix_filled = np.where(nan_mask, row_means, flux_matrix)

    # Handle any remaining NaN (rows that were all NaN)
    flux_matrix_filled = np.nan_to_num(flux_matrix_filled, nan=0.0)

    # Fit PCA
    pca = PCA(n_components=PCA_N_COMPONENTS, random_state=RANDOM_STATE)
    pca_components = pca.fit_transform(flux_matrix_filled)

    print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"Total explained variance: {sum(pca.explained_variance_ratio_):.3f}")

    # Create PCA features dataframe
    pca_df = pd.DataFrame({
        'object_id': pca_object_ids,
        'pca_1': pca_components[:, 0],
        'pca_2': pca_components[:, 1],
        'pca_3': pca_components[:, 2]
    })

    print(f"PCA features shape: {pca_df.shape}")
else:
    print("No PCA features generated.")
    pca_df = pd.DataFrame(columns=['object_id', 'pca_1', 'pca_2', 'pca_3'])

In [ ]:
# Convert feature list to dataframe
df_new_features = pd.DataFrame(all_features)
print(f"New features shape: {df_new_features.shape}")

# Merge PCA features
if not pca_df.empty:
    df_new_features = df_new_features.merge(pca_df, on='object_id', how='left')
    print(f"After PCA merge: {df_new_features.shape}")

# Merge with existing features (if available)
if not df_existing.empty:
    # Ensure string join key
    df_existing['object_id'] = df_existing['object_id'].astype(str)
    df_new_features['object_id'] = df_new_features['object_id'].astype(str)

    df_final = df_existing.merge(df_new_features, on='object_id', how='left')
    print(f"Final merged shape: {df_final.shape}")

    # Show new columns
    new_cols = [c for c in df_final.columns if c not in df_existing.columns]
    print(f"\nNew columns added ({len(new_cols)}):")
    print(new_cols)
else:
    df_final = df_new_features
    print("Using only new features as existing file was missing.")

In [ ]:
# Save to parquet
df_final.to_parquet(OUTPUT_PARQUET, index=False)
print(f"\n✅ Saved to: {OUTPUT_PARQUET}")
print(f"Final shape: {df_final.shape}")